# pandas: Interactive Tour

Where NumPy/PyTorch store numbers in arrays, pandas stores *labeled,
heterogeneous* tabular data. None of the notebooks ahead import it directly,
they work in raw tensors end to end, but the moment you're running a
sweep across many `(layer, head, prompt)` combinations and want to sort,
filter, or pivot the results instead of squinting at a raw `[n_layers,
n_heads]` grid, this is the tool you reach for.


```python
import pandas as pd
import numpy as np
```


In [ ]:
# type it yourself, then run

## Series and DataFrame

A `Series` is a single labeled 1D column. A `DataFrame` is a table of them,
think "dict of columns," each with its own dtype, sharing a common row
index.


```python
s = pd.Series([10, 20, 30], name="score")
print(s)

df = pd.DataFrame({
    "model": ["gpt2-small", "gpt2-medium", "gpt2-large", "gpt2-xl"],
    "n_layers": [12, 24, 36, 48],
    "n_params_m": [124, 355, 774, 1558],
    "induction_head": [True, True, True, False],
})
df
```


In [ ]:
# type it yourself, then run

## Selecting data

- `df["col"]` or `df.col`: one column, as a `Series`
- `df[["col1", "col2"]]`: several columns, as a `DataFrame`
- `df.loc[row_label, col_label]`: select by *label*
- `df.iloc[row_pos, col_pos]`: select by integer *position*
- `df[boolean_mask]`: filter rows, exactly like NumPy boolean indexing


```python
print(df["model"])
print(df[["model", "n_layers"]])
print(df.iloc[0])            # first row
print(df.loc[df["n_layers"] > 20])   # boolean filter, same idea as numpy
```


In [ ]:
# type it yourself, then run


## Computing new columns

Assign to a new column name like a dict key. `.apply()` runs a Python
function per row/element when a vectorized expression won't do (slower,
prefer a direct vectorized expression when one exists).


```python
df["params_per_layer"] = df["n_params_m"] / df["n_layers"]
df["size_category"] = df["n_params_m"].apply(lambda p: "small" if p < 500 else "large")
df
```


In [ ]:
# type it yourself, then run


## `groupby`: split, apply, combine

The pandas equivalent of NumPy's `axis`-based reductions, but grouped by a
*label* instead of a numeric axis. `df.groupby(col).agg(...)` is the pattern
you'll use constantly when summarizing eval results by model/dataset/seed.


```python
summary = df.groupby("size_category").agg(
    mean_layers=("n_layers", "mean"),
    count=("model", "count"),
)
summary
```


In [ ]:
# type it yourself, then run


## Merging two tables

`pd.merge(left, right, on=key)` joins two DataFrames on a shared column,
like a SQL join. Common when combining, say, a table of model configs with a
separate table of eval scores.


```python
scores = pd.DataFrame({
    "model": ["gpt2-small", "gpt2-medium", "gpt2-large"],
    "ioi_accuracy": [0.91, 0.95, 0.97],
})

merged = pd.merge(df, scores, on="model", how="left")
merged[["model", "n_layers", "ioi_accuracy"]]
```


In [ ]:
# type it yourself, then run


## From a tensor to a tidy table

Real mech interp code often ends up with a `[n_layers, n_heads]` tensor of
per-head scores. `pd.DataFrame(...)` on that raw grid gives you a table
indexed by integers on both axes; `.stack()` turns it into **long format**:
one row per `(layer, head, score)`, which is what `sort_values`,
`groupby`, and `pivot_table` all expect.


```python
scores = np.array([
    [0.12, 0.88, 0.31],
    [0.65, 0.20, 0.42],
    [0.05, 0.10, 0.97],
])  # [n_layers, n_heads]

wide = pd.DataFrame(scores)
wide.index.name = "layer"
wide.columns.name = "head"
long = wide.stack().reset_index(name="score")
long
```


In [ ]:
# type it yourself, then run

## `sort_values` and `nlargest`

`.sort_values("col", ascending=False)` reorders the whole table by a
column. `.nlargest(n, "col")` is the shortcut for "top n rows by this
column," the pandas equivalent of the flatten-plus-topk idiom you used in
the torch notebook, just without the flattening step since the data is
already long format.


```python
print(long.sort_values("score", ascending=False).head(3))
print(long.nlargest(3, "score"))
```


In [ ]:
# type it yourself, then run

## `pivot_table`: long format back to a grid

`df.pivot_table(index=, columns=, values=)` is the inverse of the long-format
conversion above. Hand it a long table and get back a `layer x head` grid,
ready for `px.imshow` or any other tool that wants a 2D array.


```python
grid = long.pivot_table(index="layer", columns="head", values="score")
grid
```


In [ ]:
# type it yourself, then run

## Checkpoint: write one line yourself

Using `long` from above, compute `top3`: the 3 rows with the highest
`score`, as a DataFrame, ordered from highest to lowest. Use `.nlargest()`.

In [ ]:
top3 = None  # YOUR CODE HERE

In [ ]:
# self-check
assert top3 is not None, "fill in `top3` above"
_expected = long.nlargest(3, "score")
assert list(top3["score"]) == list(_expected["score"]), f"got {list(top3['score'])}"
print("Checkpoint passed")

## Cheat sheet

| Want to... | Call |
|---|---|
| Build a table | `pd.DataFrame({"col": [...], ...})` |
| Read a CSV | `pd.read_csv(path)` |
| Select column(s) | `df["col"]`, `df[["col1", "col2"]]` |
| Select by label / position | `df.loc[row, col]`, `df.iloc[row, col]` |
| Filter rows | `df[df["col"] > x]` |
| New column | `df["new"] = <vectorized expression>` |
| Row-wise custom function | `df["col"].apply(fn)` (slower, prefer vectorized) |
| Group + summarize | `df.groupby("col").agg(name=("source_col", "func"))` |
| Join two tables | `pd.merge(left, right, on="key", how="left")` |
| Peek at data | `df.head()`, `df.describe()`, `df.info()` |
| Wide grid -> long `(row, col, value)` table | `df.stack().reset_index(name="value")` |
| Top n rows by a column | `df.sort_values("col", ascending=False)` or `df.nlargest(n, "col")` |
| Long table -> grid | `df.pivot_table(index=, columns=, values=)` |